In [6]:
# =========================
# 1. LOAD DATA
# =========================
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("adityamane10101/disease-symptom-description-dataset")

df = pd.read_csv(os.path.join(path, "disease_symptom_dataset.csv"))
df_desc = pd.read_csv(os.path.join(path, "symptom_Description.csv"))
df_prec = pd.read_csv(os.path.join(path, "symptom_precaution.csv"))
df_sev  = pd.read_csv(os.path.join(path, "Symptom_severity.csv"))


# =========================
# 2. NORMALIZATION FUNCTION (CRITICAL FIX)
# =========================
def normalize(text):
    return str(text).strip().lower().replace(" ", "_")


# =========================
# 3. CLEAN DATA
# =========================
df = df.fillna('')

symptom_lists = []
for _, row in df.iterrows():
    symptoms = [
        normalize(s) for s in row[1:].tolist()
        if str(s).strip() != ""
    ]
    symptom_lists.append(list(set(symptoms)))  # remove duplicates

# =========================
# 4. ENCODING
# =========================
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder

mlb = MultiLabelBinarizer()
X = mlb.fit_transform(symptom_lists)

le = LabelEncoder()
y = le.fit_transform(df['Disease'].astype(str).str.strip())

# =========================
# 5. CLEAN & APPLY SEVERITY WEIGHTS
# =========================
import numpy as np

df_sev['Symptom'] = df_sev['Symptom'].apply(normalize)

sev_dict = dict(zip(df_sev['Symptom'], df_sev['weight']))

symptom_weights = np.array([
    sev_dict.get(sym, 1.0) for sym in mlb.classes_
])

X_weighted = X.astype(float) * symptom_weights


# =========================
# 6. TRAIN-TEST SPLIT
# =========================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_weighted, y, test_size=0.2, stratify=y, random_state=42
)


# =========================
# 7. TRAIN MODEL
# =========================
import xgboost as xgb

model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)


# =========================
# 8. EVALUATION (STRONGER)
# =========================
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))


# =========================
# 9. CREATE LOOKUPS (CLEANED)
# =========================
df_desc['Disease'] = df_desc['Disease'].astype(str).str.strip()
desc_dict = dict(zip(df_desc['Disease'], df_desc['Description']))

df_prec['Disease'] = df_prec['Disease'].astype(str).str.strip()

prec_dict = {
    row['Disease']: [
        row[f'Precaution_{i}'] for i in range(1, 5)
        if pd.notna(row[f'Precaution_{i}'])
    ]
    for _, row in df_prec.iterrows()
}


# =========================
# 10. SAVE EVERYTHING
# =========================
import joblib

joblib.dump(model, 'model.pkl')
joblib.dump(mlb, 'mlb.pkl')
joblib.dump(le, 'le.pkl')
joblib.dump(symptom_weights, 'weights.pkl')
joblib.dump(desc_dict, 'desc.pkl')
joblib.dump(prec_dict, 'prec.pkl')

print("\n✅ Training complete. Models saved.")

Using Colab cache for faster access to the 'disease-symptom-description-dataset' dataset.
['abdominal_pain' 'abnormal_menstruation' 'acidity' 'acute_liver_failure'
 'altered_sensorium' 'anxiety' 'back_pain' 'belly_pain' 'blackheads'
 'bladder_discomfort' 'blister' 'blood_in_sputum' 'bloody_stool'
 'blurred_and_distorted_vision' 'breathlessness' 'brittle_nails'
 'bruising' 'burning_micturition' 'chest_pain' 'chills'
 'cold_hands_and_feets' 'coma' 'congestion' 'constipation'
 'continuous_feel_of_urine' 'continuous_sneezing' 'cough' 'cramps'
 'dark_urine' 'dehydration' 'depression' 'diarrhoea' 'dischromic__patches'
 'distention_of_abdomen' 'dizziness' 'drying_and_tingling_lips'
 'enlarged_thyroid' 'excessive_hunger' 'extra_marital_contacts'
 'family_history' 'fast_heart_rate' 'fatigue' 'fluid_overload'
 'foul_smell_of_urine' 'headache' 'high_fever' 'hip_joint_pain'
 'history_of_alcohol_consumption' 'increased_appetite' 'indigestion'
 'inflammatory_nails' 'internal_itching' 'irregular_suga

In [7]:
# 10.Download Models
from google.colab import files

files.download('model.pkl')
files.download('mlb.pkl')
files.download('le.pkl')
files.download('weights.pkl')
files.download('desc.pkl')
files.download('prec.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>